# Streamlit Fundamentals

---

In this notebook, we will learn the core concepts of **Streamlit**, the Python framework for building interactive data applications with minimal code.

We will cover:

- How Streamlit works (the execution model)
- Text, layout, and page structure
- Input widgets (sliders, dropdowns, text inputs, buttons)
- Displaying data (tables, metrics, DataFrames)
- Charts and plots (built-in and Matplotlib/Plotly integration)
- Caching for performance
- Sidebar and multi-column layouts

> ⚠️ **Note:** Streamlit apps are Python scripts (`.py` files), not Jupyter notebooks. You run them with `streamlit run app.py`. This notebook explains the concepts and shows the code patterns. Copy any snippet into a `.py` file and run it to see the result.

---

## 1. Installation and First App

In [ ]:
uv add streamlit

Create a file called `hello.py`

In [ ]:
import streamlit as st

st.title("Hello, Streamlit!")
st.write("This is my first Streamlit app.")

Run it:

In [ ]:
streamlit run hello.py

Your browser opens at `http://localhost:8501` with a live web page. That's it — two lines of Python, a full web app.

---

## 2. The Execution Model

Streamlit works differently from Flask or FastAPI. Understanding this is critical:

**Every time the user interacts with a widget (moves a slider, clicks a button, types text), Streamlit reruns the entire script from top to bottom.**

This means:

- Your script is not a server with routes, it's a **top-to-bottom script** that produces a page.
- You don't write callbacks or event handlers. You just write Python, and Streamlit re-executes it.
- **Expensive operations** (loading models, reading data) should be **cached** so they don't re-execute on every interaction.

```
User moves a slider
  → Streamlit reruns the entire .py file
  → The new slider value is used in computations
  → The page updates with new results
```

This model is simple but powerful. It's why Streamlit apps can be written in so few lines.

---

## 3. Text and Layout

Streamlit provides functions for every type of text element:

In [ ]:
st.title("Main Title")              # Largest heading
st.header("Section Header")         # h2
st.subheader("Subsection Header")   # h3
st.write("This is some regular text.") # Renders text, DataFrames, charts, etc.
st.markdown("**Bold** and *italic*") # Raw markdown
st.latex(r"y = \beta_0 + \beta_1 x") # LaTeX equations
st.code("print('hello')", language="python") # Code block
st.divider() # Horizontal line      # Horizontal line

### Columns

In [ ]:
col1, col2, col3 = st.columns(3)

with col1:
    st.metric("Accuracy", "0.97")
with col2:
    st.metric("Precision", "0.95")
with col3:
    st.metric("Recall", "0.96")

### Sidebar

In [ ]:
st.sidebar.title("Controls")
name = st.sidebar.text_input("Your name")
st.sidebar.slider("Choose a value", 0, 100, 50)

The sidebar is perfect for input controls — it keeps the main area clean for outputs and visualizations.

---

## 4. Input Widgets

Widgets are the interactive elements that make Streamlit apps dynamic. Each widget returns its current value.

In [ ]:
# Slider: returns a number
age = st.slider("Age", min_value=0, max_value=120, value=25, step=1)

# Number input: returns a number
weight = st.number_input("Weight (kg)", min_value=0.0, max_value=300.0, value=70.0, step=0.1)

# Dropdown: returns the selected string
color = st.selectbox("Favorite color", ["Red", "Blue", "Green"])

# Multi-select: returns a list of selected options
features = st.multiselect("Select features", ["sepal_length", "sepal_width", "petal_length", "petal_width"])

# Text input: returns a string
name = st.text_input("Your name", value="Ibrahim")

# Checkbox — returns True/False
show_details = st.checkbox("Show details")

# Button - returns True when clicked (only for that one rerun)
if st.button("Predict"):
    st.write("Making a prediction...")
    
# File uploader: returns a file-like object (or None)
uploaded_file = st.file_uploader("Upload a CSV", type=["csv"])

### The Key Insight 

Every widget **returns its value immediately**. You don't attach callbacks. You just use the return value in your code.

In [ ]:
threshold = st.slider("Threshold", 0.0, 1.0, 0.5)
st.write(f"Current threshold: {threshold:.2f}")

When the user moves the slider, the script reruns, threshold gets the new value, and the text updates.

---

## 5. Displaying Data

Streamlit has built-in support for DataFrames, metrics, and JSON.

In [ ]:
import pandas as pd

df = pd.DataFrame({"Species": ["setosa", "versicolor", "virginica"], "Count": [50, 50, 50]})

# Interactive sortable/filterable table
st.dataframe(df)

# Static table (no interactivity)
st.table(df)

# Metric with delta indicator
st.metric(label="Accuracy", value="97.3%", delta="+2.1%")

# JSON display
st.json({"prediction": "setosa", "confidence": 0.97})

### Status Elements

In [ ]:
st.success("Model loaded successfully!")
st.warning("Low confidence prediction.")
st.error("Failed to load model.")
st.info("Tip: Adjust the sliders to see different predictions.")


---

## 6. Charts and Plots

Streamlit supports multiple chart libraries.

### Built-in Charts

In [ ]:
import numpy as np

data = np.random.randn(100, 3)
st.line_chart(data)
st.bar_chart(data)
st.area_chart(data)

### Matplotlib / Seaborn

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.hist(np.random.randn(200), bins=20)
ax.set_title("Distribution")
st.pyplot(fig)

### Plotly

In [ ]:
import plotly.express as px

df_2 = px.data.iris()
fig = px.scatter(df_2, x="sepal_length", y="petal_length", color="species")
st.plotly_chart(fig, use_container_width=True)

Plotly charts are interactive by default (hover, zoom, pan), great for client demos.

---

## 7. Caching

Remember: Streamlit reruns your entire script on every interaction. If your script loads a model or reads a large CSV, that would happen every time the user moves a slider. **Caching** prevents this.

| **Decorator** | **Use For** | **How It Works** |
| :--- | :--- | :--- |
| `@st.cache_data` | DataFrames, lists, dicts, API responses | Returns a **copy** of the cached data (safe to mutate) |
| `st.cache_resource` | ML models, DB connections, heavy objects | Returns the **same object** (shared across reruns, don't mutate) |

In [ ]:
# Caching data
@st.cache_data
def load_data():
    return pd.read_csv("large_dataset.csv")

df = load_data()

In [ ]:
# Caching model
import joblib

@st.cache_resource
def load_model():
    return joblib.load("model.pkl")

pipe = load_model()


---

## 8. Session State

Sometimes you need to persist values across reruns beyond what widgets provide. `st.session_state` is a dictionary that survives reruns:

In [ ]:
# Initialize a counter
if "count" not in st.session_state:
    st.session_state.count = 0
    
# Increment on button click
if st.button("Increment"):
    st.session_state.count += 1
    
st.write(f"Current count: {st.session_state.count}")

Session state is useful for:

- Tracking prediction history
- Storing uploaded data across pages
- Building multi-step forms

---

## 9. Summary

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **Execution Model** | The entire script reruns on every interaction. No callbacks, no routes. |
| **Widgets** | `st.slider()`, `st.selectbox()`, etc. Each returns its current value |
| **Layout** | `st.columns()` for multi-column, `st.sidebar` for controls, `st.tabs` for tabbed content |
| **Data display** | `st.dataframe()`, `st.metric()`, `st.json()` |
| **Charts** | Built in (`st.bar_chart`), Matplotlib (`st.pyplot`), Plotly (`st.plotly_chart`) |

---

**Next:** [Building an ML Dashboard](./02_building_an_ml_dashboard.ipynb) — Putting it all together to build the Iris Prediction Dashboard.